# Stage 2: Prepare Base Model using Transfer Learning

This notebook demonstrates the preparation of a base model using transfer learning with VGG16 architecture for disease classification.

## Objectives:
1. Load pre-trained VGG16 model from ImageNet
2. Freeze base model weights
3. Add custom classification layers
4. Compile the model
5. Save both base and updated models

In [1]:
import os
import sys

In [2]:
%pwd

'e:\\data science\\Deep_learning_project\\Disease_classification_project\\research'

In [3]:
os.chdir("../")
%pwd

'e:\\data science\\Deep_learning_project\\Disease_classification_project'

In [4]:
%pwd

'e:\\data science\\Deep_learning_project\\Disease_classification_project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int

In [6]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml, create_directories

In [7]:
from cnnclassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnclassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
            
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            create_directories([self.config.artifact_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
          config = self.config.prepare_base_model

          create_directories([config.root_dir])

          prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IAMGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
            ) 
          
          return prepare_base_model_config

In [8]:
import os
import tensorflow as tf
from cnnclassifier import logger

## Step 1: Define PrepareBaseModel Component

This component handles:
1. Loading VGG16 pre-trained model
2. Freezing base weights
3. Adding custom classification layers
4. Compiling the model

In [9]:
class PrepareBaseModel:
    """
    Prepare base model using transfer learning with VGG16.
    
    This class loads a pre-trained VGG16 model from ImageNet, removes the top layers,
    freezes the base weights, adds custom layers for the specific classification task,
    and saves the model.
    """
    
    def __init__(self, config: PrepareBaseModelConfig):
        """
        Initialize PrepareBaseModel with configuration.
        
        Args:
            config (PrepareBaseModelConfig): Configuration for base model preparation
        """
        self.config = config
        logger.info(f"PrepareBaseModel initialized")

    def get_base_model(self):
        """
        Load pre-trained VGG16 model from ImageNet.
        
        Returns:
            tf.keras.Model: Pre-trained VGG16 model
        """
        logger.info(f"Loading VGG16 model with weights: {self.config.params_weights}")
        
        base_model = tf.keras.applications.vgg16.VGG16(
            input_shape=self.config.params_image_size,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )
        
        logger.info(f"Base model loaded successfully")
        return base_model

    def prepare_full_model(self, base_model, learning_rate):
        """
        Prepare the full model by adding custom top layers to the base model.
        
        This function:
        1. Freezes the base model weights
        2. Adds custom layers for classification
        3. Compiles the model
        
        Args:
            base_model (tf.keras.Model): Pre-trained base model
            learning_rate (float): Learning rate for the optimizer
            
        Returns:
            tf.keras.Model: Complete model ready for training
        """
        logger.info(f"Preparing full model with custom top layers")
        
        # Freeze the base model weights
        base_model.trainable = False
        logger.info(f"Base model weights frozen")
        
        # Create the full model by adding custom layers
        model = tf.keras.Sequential([
            base_model,
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(
                units=512,
                activation='relu',
                name='dense_1'
            ),
            tf.keras.layers.Dropout(rate=0.3, name='dropout_1'),
            tf.keras.layers.Dense(
                units=256,
                activation='relu',
                name='dense_2'
            ),
            tf.keras.layers.Dropout(rate=0.2, name='dropout_2'),
            tf.keras.layers.Dense(
                units=self.config.params_classes,
                activation='softmax',
                name='output_layer'
            )
        ])
        
        logger.info(f"Full model created with custom top layers")
        
        # Compile the model
        logger.info(f"Compiling model with learning rate: {learning_rate}")
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=['accuracy']
        )
        
        logger.info(f"Model compiled successfully")
        
        return model

    def save_model(self, model, file_path):
        """
        Save the model to the specified path.
        
        Args:
            model (tf.keras.Model): Model to save
            file_path (Path): Path where to save the model
            
        Returns:
            bool: True if saved successfully
        """
        logger.info(f"Saving model to: {file_path}")
        model.save(file_path)
        logger.info(f"Model saved successfully at: {file_path}")
        return True

    def prepare_and_save_base_model(self):
        """
        Main function to prepare and save the base model.
        """
        logger.info("=" * 80)
        logger.info("Starting Base Model Preparation Process")
        logger.info("=" * 80)
        
        try:
            # Step 1: Load base model
            logger.info("\n[STEP 1] Loading base model...")
            base_model = self.get_base_model()
            
            # Step 2: Save base model
            logger.info("\n[STEP 2] Saving base model...")
            self.save_model(
                model=base_model,
                file_path=self.config.base_model_path
            )
            
            # Step 3: Prepare full model with custom layers
            logger.info("\n[STEP 3] Preparing full model with custom layers...")
            full_model = self.prepare_full_model(
                base_model=base_model,
                learning_rate=self.config.params_learning_rate
            )
            
            # Step 4: Save updated/full model
            logger.info("\n[STEP 4] Saving updated full model...")
            self.save_model(
                model=full_model,
                file_path=self.config.updated_base_model_path
            )
            
            logger.info("\n" + "=" * 80)
            logger.info("Base Model Preparation Process Completed Successfully!")
            logger.info("=" * 80)
            
            return full_model
            
        except Exception as e:
            logger.error(f"Error in prepare_and_save_base_model: {str(e)}")
            raise e

## Step 2: Execute the Pipeline

Now let's execute the complete prepare base model pipeline.

In [11]:
try:
    # Step 1: Initialize configuration manager
    logger.info("Initializing configuration manager...")
    config = ConfigurationManager()
    
    # Step 2: Get prepare base model config
    logger.info("Getting prepare base model configuration...")
    prepare_base_model_config = config.get_prepare_base_model_config()
    
    # Step 3: Create PrepareBaseModel instance
    logger.info("Creating PrepareBaseModel instance...")
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    
    # Step 4: Execute the pipeline
    logger.info("Starting base model preparation pipeline...")
    full_model = prepare_base_model.prepare_and_save_base_model()
    
    logger.info("\nPipeline execution completed successfully!")
    
except Exception as e:
    logger.error(f"Error during pipeline execution: {str(e)}")
    raise e

[2026-05-26 17:38:09,447: INFO: 944745852: Initializing configuration manager...]

Loading YAML file: config\config.yaml
CONTENT:
{'artifact_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://raw.githubusercontent.com/Vasunavadiya90/data_zip/main/Chicken-fecal-images.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updated.h5'}}
TYPE:
<class 'dict'>
YAML loaded successfully: config\config.yaml

Loading YAML file: params.yaml
CONTENT:
{'AUGMENTATION': True, 'IAMGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLASSES': 2, 'WEIGHTS': 'imagenet', 'LEARNING_RATE': 0.01}
TYPE:
<class 'dict'>
YAML loaded successfully: params.yaml
[2026-05-26 17:38:09,453: INFO: 

## Step 3: Verify Model Creation

Let's verify that both models have been created successfully and display their summaries.

In [12]:
import os

# Verify models exist
base_model_path = prepare_base_model_config.base_model_path
updated_model_path = prepare_base_model_config.updated_base_model_path

print("\n" + "="*80)
print("MODEL VERIFICATION")
print("="*80)

if os.path.exists(base_model_path):
    base_model_size = os.path.getsize(base_model_path) / (1024 * 1024)
    print(f"✓ Base Model exists at: {base_model_path}")
    print(f"  Size: {base_model_size:.2f} MB")
else:
    print(f"✗ Base Model NOT found at: {base_model_path}")

if os.path.exists(updated_model_path):
    updated_model_size = os.path.getsize(updated_model_path) / (1024 * 1024)
    print(f"✓ Updated Model exists at: {updated_model_path}")
    print(f"  Size: {updated_model_size:.2f} MB")
else:
    print(f"✗ Updated Model NOT found at: {updated_model_path}")

print("="*80)

# Display model architecture
print("\nFull Model Architecture:")
print("="*80)
full_model.summary()

print("\nModel Configuration:")
print(f"  - Input Shape: {prepare_base_model_config.params_image_size}")
print(f"  - Number of Classes: {prepare_base_model_config.params_classes}")
print(f"  - Learning Rate: {prepare_base_model_config.params_learning_rate}")
print(f"  - Include Top: {prepare_base_model_config.params_include_top}")
print(f"  - Weights: {prepare_base_model_config.params_weights}")
print("="*80)


MODEL VERIFICATION
✓ Base Model exists at: artifacts\prepare_base_model\base_model.h5
  Size: 56.20 MB
✓ Updated Model exists at: artifacts\prepare_base_model\base_model_updated.h5
  Size: 105.71 MB

Full Model Architecture:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,692,098 (105.64 MB)

 Trainable params: 12,977,410 (49.50 MB)

 Non-trainable params: 14,714,688 (56.13 MB)


Model Configuration:
  - Input Shape: [224, 224, 3]
  - Number of Classes: 2
  - Learning Rate: 0.01
  - Include Top: False
  - Weights: imagenet
